#Content-based Reco using NLP

In [7]:
import pandas as pd

# Load the datasets
book_data = pd.read_csv('book_data_sample_ed.csv')
user_data = pd.read_csv('users_data_sample_ed.csv')
ratings_data = pd.read_csv('ratings_data_sample_ed.csv')

In [8]:
# Display the first few rows of each dataframe to understand their structure
book_data.head(10)

,book_name,title,description,genres,av_rating,num_rating,ISBN,Year-Of-Publication,Book-Author,Publisher,Image-URL-S,_merge
0,book_1,to kill a mockingbird,The unforgettable novel of a childhood in a sl...,"['Classics', 'Fiction', 'Historical Fiction', ...",4.27,5691311,0060935464,2002.0,Harper Lee,Perennial Classics,http://images.amazon.com/images/P/0060935464.0...,both
1,book_2,harry potter and the sorcerer's stone (harry p...,Harry Potter thinks he is an ordinary boy - un...,"['Fantasy', 'Fiction', 'Young Adult', 'Magic',...",4.47,9278135,0613206339,2001.0,J. K. Rowling,Sagebrush Bound,http://images.amazon.com/images/P/0613206339.0...,both
2,book_3,pride and prejudice,"Since its immediate success in 1813, Pride and...","['Classics', 'Fiction', 'Romance', 'Historical...",4.28,3944155,3190025355,1999.0,Jane Austen,Max Hueber Verlag,http://images.amazon.com/images/P/3190025355.0...,both
3,book_4,the diary of a young girl,Discovered in the attic in which she spent the...,"['Classics', 'Nonfiction', 'History', 'Biograp...",4.18,3488438,0671461990,1982.0,Anne Frank,Washington Square Press,http://images.amazon.com/images/P/0671461990.0...,both
4,book_5,animal farm,Librarian's note: There is an Alternate Cover ...,"['Classics', 'Fiction', 'Dystopia', 'Fantasy',...",3.98,3575172,1854597892,2004.0,George Orwell,Nick Hern Books,http://images.amazon.com/images/P/1854597892.0...,both
5,book_6,the little prince,A pilot stranded in the desert awakes one morn...,"['Classics', 'Fiction', 'Fantasy', 'Childrens'...",4.32,1924063,0152023984,2000.0,Antoine de Saint-ExupÃ©ry,Harcourt Children's Books,http://images.amazon.com/images/P/0152023984.0...,both
6,book_7,1984,The new novel by George Orwell is the major wo...,"['Classics', 'Fiction', 'Science Fiction', 'Dy...",4.19,4201429,8423328651,1998.0,George Orwell,La Marca Editora,http://images.amazon.com/images/P/8423328651.0...,both
7,book_8,the great gatsby,Alternate Cover Edition ISBN: 0743273567 (ISBN...,"['Classics', 'Fiction', 'School', 'Historical ...",3.93,4839642,185326041X,1999.0,F. Scott Fitzgerald,Lb May &amp; Assoc Inc,http://images.amazon.com/images/P/185326041X.0...,both
8,book_9,the catcher in the rye,It's Christmas time and Holden Caulfield has j...,"['Classics', 'Fiction', 'Young Adult', 'Litera...",3.81,3315881,0316769177,2001.0,J.D. Salinger,Back Bay Books,http://images.amazon.com/images/P/0316769177.0...,both
9,book_10,the lord of the rings,"One Ring to rule them all, One Ring to find th...","['Fantasy', 'Classics', 'Fiction', 'Adventure'...",4.52,644766,0618343997,2003.0,J. R. R. Tolkien,Houghton Mifflin Company,http://images.amazon.com/images/P/0618343997.0...,both


In [9]:
# Display the first few rows of each dataframe to understand their structure
user_data.head(10)

,User-ID,Location,Age
0,99,"franktown, colorado, usa",42.0
1,507,"dumas, arkansas, usa",NaN
2,1424,"north avoca, new south wales, australia",NaN
3,1848,"bellingham, washington, usa",22.0
4,2110,"charlotte, north carolina, usa",NaN
5,2982,"montreal, quebec, canada",34.0
6,3332,"hartlepool, cleveland, united kingdom",54.0
7,4017,"new orleans, louisiana, usa",48.0
8,4385,"albq, new mexico, usa",33.0
9,4599,"glendale, california, usa",NaN


In [10]:
# Display the first few rows of each dataframe to understand their structure
ratings_data.head(10)

,User-ID,ISBN,Book-Rating
0,277478,0786868716,0
1,278418,0140361219,0
2,278418,0394800168,10
3,278422,0060935464,9
4,99,0786868716,0
5,507,0064400026,0
6,1424,0060512822,7
7,1424,0786868716,0
8,1848,0786868716,10
9,2110,0786868716,0


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, MultiLabelBinarizer
import numpy as np
import ast

# Preprocess the genres
book_data['genres'] = book_data['genres'].apply(ast.literal_eval)  # Convert stringified lists to actual lists
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(book_data['genres'])

# TF-IDF Vectorization of descriptions
tfidf_vectorizer_desc = TfidfVectorizer(stop_words='english')
tfidf_desc = tfidf_vectorizer_desc.fit_transform(book_data['description'])

# Normalize 'av_rating' and 'Year-Of-Publication'
book_data['Year-Of-Publication'] = book_data['Year-Of-Publication'].fillna(book_data['Year-Of-Publication'].median())  # Fill NaNs
ratings_normalized = normalize(np.array(book_data['av_rating']).reshape(-1, 1))
years_normalized = normalize(np.array(book_data['Year-Of-Publication']).reshape(-1, 1))

# Stack all features together
from scipy.sparse import hstack
features = hstack([genres_encoded, tfidf_desc, ratings_normalized, years_normalized])

features.shape


(100, 3635)

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
cosine_sim = cosine_similarity(features, features)

# Convert to DataFrame for easier handling
cosine_sim_df = pd.DataFrame(cosine_sim, index=book_data['book_name'], columns=book_data['book_name'])

def get_recommendations(book_name, cosine_sim_df, book_data, top_n=10):
    # Get the top_n most similar books
    sim_scores = cosine_sim_df[book_name].sort_values(ascending=False)
    top_books = sim_scores.iloc[1:top_n+1].index  # Exclude the book itself

    # Return the top_n most similar books, including details and sorted by num_rating
    recommended_books = book_data.loc[book_data['book_name'].isin(top_books)].sort_values('num_rating', ascending=False)
    return recommended_books[['book_name', 'title', 'Book-Author', 'av_rating', 'num_rating']]

# Example: Get 10 most similar books to "book_5"
recommendations = get_recommendations('book_5', cosine_sim_df, book_data)
recommendations

,book_name,title,Book-Author,av_rating,num_rating
6,book_7,1984,George Orwell,4.19,4201429
13,book_14,lord of the flies,William Golding,3.69,2738919
17,book_18,the giver,LOIS LOWRY,4.12,2285401
24,book_25,fahrenheit 451,Ray Bradbury,3.97,2215889
73,book_74,the old man and the sea,Ernest Hemingway,3.80,1060714
47,book_48,the odyssey,Homer,3.80,1017517
79,book_80,the metamorphosis,PETER KUPER,3.85,918431
60,book_61,a tale of two cities,Charles Dickens,3.86,911758
52,book_53,the grapes of wrath,John Steinbeck,4.00,864334
41,book_42,brave new world,Aldous Huxley,4.16,159408
